# 02 — Data Cleaning

**Goal:** Filter Nordic countries, average Men/Women into a single estimate, 
merge diabetes and obesity datasets into one clean file.

**Output:** `data/nordic_cleaned.csv` — 198 rows × 6 columns, 0 missing values.

In [1]:
import sys
sys.path.append('..')

from src.load_data import load_diabetes, load_obesity
from src.clean_data import get_nordic_both_sexes, merge_obesity

df_diabetes = load_diabetes()
df_obesity = load_obesity()
df_nordic = get_nordic_both_sexes(df_diabetes)
df = merge_obesity(df_nordic, df_obesity)

In [2]:
# Rename columns
df_diabetes = df_diabetes.rename(columns={
    'Country/Region/World': 'country',
    'ISO': 'iso',
    'Sex': 'sex',
    'Year': 'year',
    'Age': 'age',
    'Prevalence of diabetes (18+ years)': 'diabetes_prevalence',
    'Prevalence of diabetes (18+ years) lower 95% uncertainty interval': 'diabetes_lower_ci',
    'Prevalence of diabetes (18+ years) upper 95% uncertainty interval': 'diabetes_upper_ci',
    'Proportion of people with diabetes who were treated (30+ years)': 'treatment_rate',
    'Proportion of people with diabetes who were treated (30+ years) lower 95% uncertainty interval': 'treatment_lower_ci',
    'Proportion of people with diabetes who were treated (30+ years) upper 95% uncertainty interval': 'treatment_upper_ci'
})

df_obesity = df_obesity.rename(columns={
    'Entity': 'country',
    'Code': 'iso',
    'Year': 'year',
    'Prevalence of obesity among adults, BMI >= 30 (crude estimate) (%) - Sex: both sexes - Age group: 18+  years of age': 'obesity_prevalence'
})

print("Loaded successfully.")

Loaded successfully.


In [3]:
nordic = ['Sweden', 'Norway', 'Denmark', 'Finland', 'Iceland', 'Netherlands']

# Filter Nordic, then average across sexes
df_nordic = (
    df_diabetes[df_diabetes['country'].isin(nordic)]
    .groupby(['country', 'iso', 'year'], as_index=False)
    [['diabetes_prevalence', 'treatment_rate']]
    .mean()
)

print("Shape:", df_nordic.shape)
df_nordic.head(6)

Shape: (198, 5)


,country,iso,year,diabetes_prevalence,treatment_rate
0,Denmark,DNK,1990,0.026467,0.318880
1,Denmark,DNK,1991,0.027161,0.323102
2,Denmark,DNK,1992,0.027819,0.327693
3,Denmark,DNK,1993,0.028435,0.332619
4,Denmark,DNK,1994,0.028977,0.337866
5,Denmark,DNK,1995,0.029424,0.343484


### Merge obesity data
Left join on country + year. Obesity data covers 1990–2016; 
years 2017–2022 will have NaN for obesity_prevalence.

In [5]:
import pandas as pd

df_obesity_nordic = df_obesity[df_obesity['country'].isin(nordic)]

df_merged = pd.merge(df_nordic, df_obesity_nordic[['country', 'year', 'obesity_prevalence']],
                     on=['country', 'year'], how='left')

print("Merged shape:", df_merged.shape)
print("Missing obesity values:", df_merged['obesity_prevalence'].isna().sum())
df_merged.head()

Merged shape: (198, 6)
Missing obesity values: 0


,country,iso,year,diabetes_prevalence,treatment_rate,obesity_prevalence
0,Denmark,DNK,1990,0.026467,0.318880,8.57632
1,Denmark,DNK,1991,0.027161,0.323102,8.82850
2,Denmark,DNK,1992,0.027819,0.327693,9.09095
3,Denmark,DNK,1993,0.028435,0.332619,9.36231
4,Denmark,DNK,1994,0.028977,0.337866,9.64552


In [6]:
df_merged.to_csv('../data/nordic_cleaned.csv', index=False)
print("Saved to ../data/nordic_cleaned.csv")

Saved to ../data/nordic_cleaned.csv
